### Data validation tools
Because bad data in means bad model out, and you won't always know when it happens.

---
## Setup
Install the tools we'll use. `pandera` and `evidently` are not in the base environment.

In [ ]:
# Install validation libraries if not already present
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "pandera", "evidently", "great-expectations", "--quiet"])
print("Done")

In [ ]:
import pandas as pd
import numpy as np

# Load the iris dataset
df = pd.read_csv("../Session_1/iris.csv")
print(df.shape)
df.head()

In [ ]:
# Quick overview: types, null counts, basic stats
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nValue counts for class:")
print(df["class"].value_counts())

---
## Part 1 — Pandera: schema validation on a DataFrame

Pandera lets you declare what a valid DataFrame looks like as a typed schema,
then call `.validate()` to enforce it. Validation errors raise a `SchemaError`
immediately, which is exactly what you want at the top of a training script.

**Best for:** lightweight in-code checks before training or feature engineering.

In [ ]:
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check

# Define the expected schema for the iris dataset
iris_schema = DataFrameSchema(
    columns={
        "septal_length": Column(float, Check.ge(0.0), nullable=False),
        "sepal_width":   Column(float, Check.ge(0.0), nullable=False),
        "petal_length":  Column(float, Check.ge(0.0), nullable=False),
        "petal_width":   Column(float, Check.ge(0.0), nullable=False),
        "class":         Column(str,   Check.isin([
                                     "Iris-setosa",
                                     "Iris-versicolor",
                                     "Iris-virginica"
                                 ]), nullable=False),
    }
)

# Validate the real dataset — should pass cleanly
validated_df = iris_schema.validate(df)
print("Validation passed. Shape:", validated_df.shape)

In [ ]:
# Introduce a deliberate schema violation and observe the error
bad_df = df.copy()
bad_df.loc[0, "septal_length"] = -1.0   # negative measurement — physically impossible
bad_df.loc[5, "class"] = "Iris-unknown" # unexpected class label

try:
    iris_schema.validate(bad_df, lazy=True)  # lazy=True collects ALL errors before raising
except pa.errors.SchemaErrors as e:
    print("Schema violations found:")
    print(e.failure_cases)

In [ ]:
# Pandera also works as a decorator on functions — useful for validating
# the output of a feature engineering step
from pandera.typing import DataFrame, Series

class IrisSchema(pa.DataFrameModel):
    septal_length: Series[float] = pa.Field(ge=0.0)
    sepal_width:   Series[float] = pa.Field(ge=0.0)
    petal_length:  Series[float] = pa.Field(ge=0.0)
    petal_width:   Series[float] = pa.Field(ge=0.0)

@pa.check_output(IrisSchema)
def drop_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the class column, returning only numeric features."""
    return df.drop(columns=["class"])

features = drop_labels(df)
print("Output validated by decorator. Shape:", features.shape)
features.head(3)

**Key observations:**
- `lazy=True` collects all violations before raising, which is useful in CI — you get the full list rather than stopping at the first failure.
- The `DataFrameModel` + decorator pattern enforces the contract on the *output* of a function, not just a loaded file.
- Pandera does not produce a report. If you need a shareable HTML artefact, use Great Expectations.

---
## Part 2 — Great Expectations: expectation suites and validation reports

Great Expectations (GX) builds a suite of "expectations" — declarative assertions —
about your data, then runs them and produces an HTML report. The report is the main
value-add over Pandera: it's shareable with non-technical stakeholders and can be
committed as a CI artefact.

**Best for:** CI/CD data quality gates; audit trails; team-wide data contracts.

In [ ]:
import great_expectations as gx

print("GX version:", gx.__version__)

In [ ]:
# GX fluent API (v1+): create an in-memory data source from a pandas DataFrame
context = gx.get_context(mode="ephemeral")

data_source = context.data_sources.add_pandas(name="iris_source")
data_asset  = data_source.add_dataframe_asset(name="iris")
batch_def   = data_asset.add_batch_definition_whole_dataframe("iris_batch")
batch       = batch_def.get_batch(batch_parameters={"dataframe": df})

print("Batch created:", batch)

In [ ]:
# Build an expectation suite — these are the assertions about what valid data looks like
suite = context.suites.add(gx.ExpectationSuite(name="iris_suite"))

# Column presence and types
suite.add_expectation(gx.expectations.ExpectTableColumnsToMatchOrderedList(
    column_list=["septal_length", "sepal_width", "petal_length", "petal_width", "class"]
))
suite.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=100, max_value=200))

# Value range checks on numeric columns
for col in ["septal_length", "sepal_width", "petal_length", "petal_width"]:
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
        column=col, min_value=0.0, max_value=50.0
    ))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))

# Class column
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column="class",
    value_set=["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
))

print(f"Suite '{suite.name}' has {len(suite.expectations)} expectations")

In [ ]:
# Run validation and inspect results
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(name="iris_validation", data=batch_def, suite=suite)
)
results = validation_def.run(batch_parameters={"dataframe": df})

print(f"Overall success: {results.success}")
print(f"Results: {results.describe_dict()['statistics']}")

In [ ]:
# Now run against the bad_df from Part 1 — some expectations should fail
results_bad = validation_def.run(batch_parameters={"dataframe": bad_df})

print(f"Overall success: {results_bad.success}")
for r in results_bad.results:
    if not r.success:
        print(f"  FAILED: {r.expectation_config.type} on column '{r.expectation_config.column if hasattr(r.expectation_config, 'column') else 'N/A'}'")

**Key observations:**
- GX separates *defining* expectations (done once, versioned) from *running* them (done every pipeline run).
- The same suite can be run against different batches — last week's data, today's data, a staging table.
- In a real pipeline you'd checkpoint this and fail the step if `results.success == False`.
- Trade-off vs Pandera: more setup, but the expectation suite is a first-class artefact that non-engineers can read and contribute to.

---
## Part 3 — Evidently: distribution drift between two dataset splits

Evidently compares a **reference** dataset (e.g. training data) against a **current**
dataset (e.g. last week's production traffic) and reports where the distributions
have shifted. This is the tool for **monitoring**, not for blocking a pipeline step.

We'll simulate drift by splitting iris into two halves and then introducing
a synthetic shift in one column.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

# Split: first 100 rows = reference (training), last 50 = current (production)
reference = df.iloc[:100].copy()
current   = df.iloc[100:].copy()

print(f"Reference: {reference.shape}, Current: {current.shape}")

In [ ]:
# Run a drift report — no drift expected since both halves come from the same dataset
r1 = Report(metrics=[DataDriftPreset()])
snapshot1 = r1.run(reference_data=reference, current_data=current)

d1 = snapshot1.dict()
for m in d1["metrics"]:
    if m["metric_name"].startswith("DriftedColumnsCount"):
        count = int(m["value"]["count"])
        total = len(df.columns)
        print(f"Dataset drift detected: {count > 0}")
        print(f"Drifted columns:        {count} / {total}")

In [ ]:
# Now simulate drift: shift the sepal_width distribution in 'current' by +2.0
# This mimics a sensor recalibration or upstream data pipeline change
current_drifted = current.copy()
current_drifted["sepal_width"] = current_drifted["sepal_width"] + 2.0

print("Reference sepal_width mean:", reference["sepal_width"].mean().round(3))
print("Drifted  sepal_width mean: ", current_drifted["sepal_width"].mean().round(3))

In [ ]:
# Run drift report against the drifted current data
r2 = Report(metrics=[DataDriftPreset()])
snapshot2 = r2.run(reference_data=reference, current_data=current_drifted)

d2 = snapshot2.dict()
for m in d2["metrics"]:
    if m["metric_name"].startswith("DriftedColumnsCount"):
        count = int(m["value"]["count"])
        total = len(df.columns)
        print(f"Dataset drift detected: {count > 0}")
        print(f"Drifted columns:        {count} / {total}")
    elif m["metric_name"].startswith("ValueDrift(column="):
        col = m["metric_name"].split("column=")[1].split(",")[0]
        pval = m["value"]
        drifted = pval < 0.05
        flag = "<-- DRIFT" if drifted else ""
        print(f"  {col:20s}  p_value={pval:.4f}  {flag}")

In [ ]:
# Save the drift report as an HTML file for sharing
snapshot2.save_html("drift_report.html")
print("Report saved to drift_report.html")
# Open this file in a browser to see the interactive Evidently dashboard

**Key observations:**
- Evidently detected drift in `sepal_width` after we shifted it by +2.0. The other columns were clean.
- The drift score is computed using statistical tests (Kolmogorov-Smirnov for numerical columns by default).
- The HTML report is designed to be opened in a browser — it includes distribution plots for every column so you can *see* how the distributions shifted, not just that they did.
- Unlike Pandera and GX, Evidently does not raise an exception. It produces a report. In a production monitoring pipeline you would inspect `drift_detected` programmatically and trigger an alert or a retraining job.